In [1]:
import os
os.chdir('./../')

In [2]:
import timm
import torch
import torch.nn as nn
import torch.nn.functional as F
import os

/home/wisc/maheswararao/code/lightning-hydra-template/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
device = 'cuda:0'

In [4]:
print(os.path.exists('artifacts/model-4d9a70bz:v1/model.ckpt'))

True


Makefile	    env_example       requirements.txt	test_plots
README.md	    environment.yaml  scripts		tests
README.md-template  example.db	      setup.py		timm_cache_dir
artifacts	    lightning_logs    slurm-19.out	uv.lock
checkpoints	    logs	      slurm-20.out	wandb
configs		    multirun	      slurm_logs
csv		    notebooks	      src
data		    pyproject.toml    temp_data_cache


In [5]:
import torch
from src.models.timm_classification_models import TimmDropOutBasicClassifier
# from src.models.timm_classification_lit_module import TimmClassificationLitModule

# Initialize model
model = TimmDropOutBasicClassifier(model_name='resnet50', num_classes=8, pretrained=True, in_chans=3, drop_rate=0.2)


CKPT_PATH="artifacts/model-9d0p731i:v1/model.ckpt"
# # ckpt_path = 'artifacts/model-bpsugy0j:v1/model.ckpt'
state_dict = torch.load(CKPT_PATH, map_location='cuda', weights_only=False)  # map_location prevents CUDA issues

# # Load weights into model
model.load_state_dict(state_dict, strict=False)


_IncompatibleKeys(missing_keys=['model.conv1.weight', 'model.bn1.weight', 'model.bn1.bias', 'model.bn1.running_mean', 'model.bn1.running_var', 'model.layer1.0.conv1.weight', 'model.layer1.0.bn1.weight', 'model.layer1.0.bn1.bias', 'model.layer1.0.bn1.running_mean', 'model.layer1.0.bn1.running_var', 'model.layer1.0.conv2.weight', 'model.layer1.0.bn2.weight', 'model.layer1.0.bn2.bias', 'model.layer1.0.bn2.running_mean', 'model.layer1.0.bn2.running_var', 'model.layer1.0.conv3.weight', 'model.layer1.0.bn3.weight', 'model.layer1.0.bn3.bias', 'model.layer1.0.bn3.running_mean', 'model.layer1.0.bn3.running_var', 'model.layer1.0.downsample.0.weight', 'model.layer1.0.downsample.1.weight', 'model.layer1.0.downsample.1.bias', 'model.layer1.0.downsample.1.running_mean', 'model.layer1.0.downsample.1.running_var', 'model.layer1.1.conv1.weight', 'model.layer1.1.bn1.weight', 'model.layer1.1.bn1.bias', 'model.layer1.1.bn1.running_mean', 'model.layer1.1.bn1.running_var', 'model.layer1.1.conv2.weight', 'mo

In [ ]:
from torch.nn.modules.dropout import _DropoutNd
model = timm.create_model('resnet50', pretrained=True, num_classes=4, in_chans=3)
in_features = model.get_classifier().in_features if hasattr(model.get_classifier(), 'in_features') else model.fc.in_features
model.fc = nn.Sequential(
    nn.Dropout(p=0.2),
    nn.Linear(in_features, 4)
)

In [ ]:
batch_size = 2
in_chans = 3
height = width = 224
x = torch.randn(batch_size, in_chans, height, width)

model(x).shape

In [ ]:
# sanity check: list dropout layers
for m in model.modules():
    if isinstance(m, nn.Dropout):
        print(m)  # should now print Dropout(p=0.2, ...)

In [ ]:
class TimmBasicClassifier(nn.Module):
    def __init__(self, model_name, num_classes, pretrained=True, in_chans=3, drop_rate=0.2):
        super().__init__()
        self.num_classes = num_classes
        self.backbone = timm.create_model(
            model_name,
            pretrained=pretrained,
            num_classes=num_classes,
            in_chans=in_chans,
            drop_rate=drop_rate   # <-- IMPORTANT for MC dropout
        )
    
    def forward(self, x):
        return self.backbone(x)

model_name = "resnet50"
num_classes = 8
basic_classifier = TimmBasicClassifier(model_name, num_classes, pretrained=True, in_chans=3, drop_rate=0.2).to(device)

In [ ]:
from torch.nn.modules.dropout import _DropoutNd

def enable_mc_dropout(model: nn.Module):
    """
    Put the whole model in eval() (freeze BN etc.),
    then selectively set ONLY dropout layers to train() so they stay stochastic.
    """
    model.eval()
    for m in model.modules():
        if isinstance(m, _DropoutNd):  # catches Dropout, Dropout2d, Dropout3d
            m.train()

@torch.no_grad()
def mc_predict_proba(model: nn.Module, x: torch.Tensor, T: int = 30):
    """
    For classification:
    - Runs T stochastic forward passes with dropout on.
    - Returns predictive mean probabilities, per-class variance,
      predictive entropy, and mutual information (BALD).
    """
    enable_mc_dropout(model)

    probs = []
    for _ in range(T):
        logits = model(x)
        probs.append(F.softmax(logits, dim=-1))
    probs = torch.stack(probs, dim=0)              # [T, B, C]

    mean_prob = probs.mean(dim=0)                  # [B, C]
    var_prob  = probs.var(dim=0, unbiased=False)   # [B, C]

    # Predictive entropy H[ E[p(y|x,w)] ]
    eps = 1e-12
    pred_entropy = -(mean_prob * (mean_prob.add(eps).log())).sum(dim=-1)   # [B]

    # Expected entropy E[ H[p(y|x,w)] ]
    exp_entropy = - (probs * (probs.add(eps).log())).sum(dim=-1).mean(dim=0)  # [B]

    # Mutual Information (BALD): H[p_bar] - E[H[p]]
    mutual_info = pred_entropy - exp_entropy                                   # [B]

    return {
        "mean_proba": mean_prob,      # average class probabilities (use for prediction)
        "var_proba": var_prob,        # per-class variance (uncertainty per class)
        "pred_entropy": pred_entropy, # overall predictive uncertainty
        "mutual_info": mutual_info,   # epistemic uncertainty proxy
    }

In [ ]:
batch_size = 2
in_chans = 3
height = width = 224
x = torch.randn(batch_size, in_chans, height, width).to(device)

res = mc_predict_proba(basic_classifier, x, T = 30)